In [1]:
# !pip install --upgrade waldiez[stuio,jupyter]

In [2]:
# Run a flow
import waldiez
from pathlib import Path
from waldiez import WaldiezRunner

import pandas as pd
import os
import json
import numpy as np
from collections import Counter

from datetime import datetime

import re
import ast

In [3]:
assert waldiez.__version__=='0.7.1'

In [5]:
import requests

# 1. Download the puzzles
puzzles_url = "https://raw.githubusercontent.com/openpipe/art/main/examples/data/temporal-clue/puzzles.json"
puzzles_response = requests.get(puzzles_url)
df = pd.read_json(puzzles_response.text)

# 2. Extract the 10 specific assignment cases
examples = [1052, 1230, 2372, 1182, 2279, 2680, 2176, 2973, 2145, 2799]
train_puzzles = df[df.index.isin(examples)].copy()

# 3. Print all solutions for reference
print("📜 --- GROUND TRUTH SOLUTIONS ---")
for i, row in train_puzzles.iterrows():
    # Ensure solution is a dict (if it was a string in JSON)
    sol = row['solution']
    if isinstance(sol, str):
        sol = json.loads(sol.replace("'", '"'))
    print(f"Case {i}: {sol}")
print("----------------------------------\n")

📜 --- GROUND TRUTH SOLUTIONS ---
Case 1052: {'A': 'Monsieur Brunette', 'B': 'Knife', 'C': 'Fountain', 'D': '11:00 PM', 'E': 'Hatred', 'F': 'Ballroom', 'G': 'Ballroom'}
Case 1182: {'A': 'Mrs. White', 'B': 'Hall', 'C': '12:30 AM', 'D': 'Hatred', 'E': 'Conservatory', 'F': 'Trophy Room'}
Case 1230: {'A': 'Mrs. White', 'B': 'Wrench', 'C': 'Fountain', 'D': 'Fear', 'E': 'Billiard Room', 'F': 'Monsieur Brunette', 'G': 'Conservatory', 'H': 'Kitchen'}
Case 2145: {'A': 'Miss Scarlet', 'B': 'Revolver', 'C': 'Dining Room', 'D': '11:00 PM', 'E': 'Fear', 'F': 'Dining Room', 'G': 'Library', 'H': 'Conservatory'}
Case 2176: {'A': 'Miss Peach', 'B': 'Billiard Room', 'C': '11:45 PM', 'D': 'Pride', 'E': 'Fountain', 'F': 'Fountain'}
Case 2279: {'A': 'Mrs. White', 'B': 'Candlestick', 'C': 'Trophy Room', 'D': 'Pride', 'E': 'Trophy Room', 'F': 'Trophy Room', 'G': 'Studio'}
Case 2372: {'A': 'Mrs. Peacock', 'B': 'Lead Pipe', 'C': 'Billiard Room', 'D': '12:00 AM', 'E': 'Anger', 'F': 'Library', 'G': 'Ballroom', 'H

/var/folders/19/yrxdvc_56111j5gv4m_wmd4r0000gn/T/ipykernel_21977/3850271368.py:6: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_json(puzzles_response.text)


In [6]:
# flow_path = "clue_test.waldiez"
# output_path = "clue_test.py"

flow_path = "clue.waldiez"
output_path = "clue.py"

temp_file_path = "waldiez_out/Waldiez_flow/latest/temp_answer.json"

In [7]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_filename = f"cases_report_{timestamp}.csv"

NUM_RUNS = 10
header_str = f"{os.path.basename(flow_path)}\nBest results after {NUM_RUNS} runs\n"

results = []

print(f"Starting fresh run. Saving to: {csv_filename}")

for i, row in train_puzzles.iterrows():
# for i, row in train_puzzles.loc[[1052]].iterrows():
    quiz_text = str(row['prompt']) 
    correct_sol = row['solution']
    
    if isinstance(correct_sol, str):
        correct_sol = ast.literal_eval(correct_sol)
    
    total_q = len(correct_sol)
    case_scores = []
    all_failed_keys = []

    print(f"\nCASE {i}: Running {NUM_RUNS} trials...")

    for run_num in range(1, NUM_RUNS + 1):
        if os.path.exists(temp_file_path):
            os.remove(temp_file_path)
        if os.path.exists("temp_answer.json"):
            os.remove("temp_answer.json")

        runner = WaldiezRunner.load(Path(flow_path))
        runner.run(output_path=output_path, message=quiz_text)

        agent_answer = None
        if os.path.exists(temp_file_path):
            try:
                with open(temp_file_path, "r") as f:
                    raw_content = json.load(f).get("answer")
                
                if isinstance(raw_content, dict):
                    agent_answer = raw_content
                elif isinstance(raw_content, str):
                    start, end = raw_content.find('{'), raw_content.rfind('}') + 1
                    if start != -1 and end != 0:
                        agent_answer = ast.literal_eval(raw_content[start:end])
            except Exception as e:
                print(f"      Parsing Error in Trial {run_num}: {e}")

        trial_score = 0
        trial_failed = []
        for key, truth_val in correct_sol.items():
            if isinstance(agent_answer, dict):
                agent_val = str(agent_answer.get(key, "MISSING")).strip().lower()
                expected_val = str(truth_val).strip().lower()
                
                if agent_val == expected_val:
                    trial_score += 1
                else:
                    trial_failed.append(key)
            else:
                trial_failed.append(key)
        
        case_scores.append(trial_score)
        all_failed_keys.append(trial_failed)
        print(f"  Trial {run_num}: {trial_score}/{total_q} Correct")
        # print(f"  Trial {run_num}: {trial_score}/{total_q} Correct. Expected: {correct_sol}")

    # --- 3. BEST SCORE CALCULATION ---
    best_score = int(max(case_scores))
    best_accuracy = (best_score / total_q) * 100
    formatted_percent = f"{best_accuracy:.2f}".replace('.', ',')
    
    flat_fails = [item for sublist in all_failed_keys for item in sublist]
    persistent_fails_list = [k for k, count in Counter(flat_fails).items() if count == NUM_RUNS]
    common_fails = ", ".join(persistent_fails_list)

    # --- 4. DATA ROW CONSTRUCTION ---
    entry = {
        "case_id": i,
        "best_score": f"{best_score}/{total_q}",
        "best_accuracy_percent": formatted_percent,
        "persistent_fails": common_fails if common_fails else "None",
        "min_score": min(case_scores),
        "max_score": max(case_scores)
    }
    
    results.append(entry)
    print(f"  CASE {i} SUMMARY: Best {best_score}/{total_q} | Range: [{min(case_scores)}-{max(case_scores)}]")

# --- 5. FINAL EXPORT ---
if results:
    results_df = pd.DataFrame(results)
    
    with open(csv_filename, "w") as f:
        f.write(header_str)
        results_df.to_csv(f, index=False)

    # Dynamic accuracy line using the specified lambda logic
    accuracy_val = results_df['best_score'].apply(lambda x: eval(x)).mean() * 100
    accuracy_str = f"Global Median Accuracy: {accuracy_val:.2f}%".replace('.', ',')

    with open(csv_filename, "a") as f:
        f.write("\n")
        f.write(accuracy_str)

    print(f"\n" + "="*40)
    print(f"CASES REPORT GENERATED: {csv_filename}")
    print(accuracy_str)
    print("="*40)

Starting fresh run. Saving to: cases_report_20260218_003452.csv

CASE 1052: Running 10 trials...
[INFO] 2026-02-17 22:34:52 [venv_waldiez/lib/python3.12/site-packages/waldiez/running/base_runner.py:272] Preparing workflow file: clue.py
[INFO] 2026-02-17 22:34:56 [waldiez/exporting/flow/orchestrator.py:89] Exporting tools ...
[INFO] 2026-02-17 22:34:56 [waldiez/exporting/flow/orchestrator.py:105] Exporting models ...
[INFO] 2026-02-17 22:34:56 [waldiez/exporting/flow/orchestrator.py:120] Exporting chats ...
[INFO] 2026-02-17 22:34:56 [waldiez/exporting/flow/orchestrator.py:134] Exporting agents ...
<Waldiez> - Starting workflow...
{"participants":[{"id":"wa-1766336188094gPnsD_atmbOK-_1Mw832g","name":"User","humanInputMode":"ALWAYS","agentType":"user_proxy"},{"id":"wa-17664232271072Ool2p2Zn32lngKDWliWF","name":"Forensics","humanInputMode":"NEVER","agentType":"assistant"},{"id":"wa-1766472069823cbEaEmYlARp-7T9MNk9vy","name":"Detective","humanInputMode":"NEVER","agentType":"assistant"},{"i